# 01 — BRONZE: Setup + Minimal Validation (Delta)

**Goal:** Initialize project schemas and log foundational Bronze quality checks.

**Creates:**
- `bronze`, `silver`, `gold`, `dq` databases
- `dq.check_results` (Delta log table)

**Validates:**
- Bronze table is not empty
- No nulls in key identifiers (`PatientId`, `AppointmentID`)


In [0]:
%sql
-- Creating schemas in Hive metastore
CREATE DATABASE IF NOT EXISTS bronze;
CREATE DATABASE IF NOT EXISTS silver;
CREATE DATABASE IF NOT EXISTS gold;
CREATE DATABASE IF NOT EXISTS dq;

-- DQ results log table(Delta)
CREATE TABLE IF NOT EXISTS dq.check_results (
  run_time_stamp TIMESTAMP,
  check_name STRING,
  status STRING,
  failed_count BIGINT,
  details STRING
)
USING DELTA;

## Minimal Bronze DQ Checks (log to `dq.check_results`)

These checks validate Bronze integrity before Silver transformations begin.


In [0]:
from pyspark.sql import functions as F

bronze_data = spark.table("bronze.appointments_raw")

def log_check(check_name: str, failed_count: int, details: str = ""):
    status = "PASS" if failed_count == 0 else "FAIL"
    (spark.createDataFrame([(check_name, status, int(failed_count), details)],
                           "check_name STRING, status STRING, failed_count BIGINT, details STRING")
          .withColumn("run_time_stamp", F.current_timestamp())
          .select("run_time_stamp", "check_name", "status", "failed_count", "details")
          .write.mode("append").format("delta").saveAsTable("dq.check_results"))

# Check 1: Bronze is not empty
empty_failed = 1 if bronze_data.limit(1).count() == 0 else 0
log_check("bronze_rowcount_gt_0", empty_failed, "bronze.appointments_raw should not be empty")

# Check 2: No null IDs in key fields
null_id_failed = bronze_data.filter(
    F.col("PatientId").isNull() | F.col("AppointmentID").isNull()
).count()
log_check("bronze_no_null_patient_or_appt", null_id_failed, "PatientId and AppointmentID must not be null")


## Quick Sanity Checks

Confirm Bronze exists and inspect recent DQ logs.


In [0]:
%sql
SELECT COUNT(*) AS bronze_rows
FROM bronze.appointments_raw;


In [0]:
%sql
SELECT *
FROM dq.check_results
ORDER BY run_time_stamp DESC
LIMIT 10;
